In [30]:
import pandas as pd
import ast
from collections import Counter
from sklearn.preprocessing import StandardScaler
from IPython.display import display

In [11]:
costumer_info= pd.read_csv("../data/customer_info_cleaned.csv", sep=',')
basket = pd.read_csv("../data/customer_basket.csv", sep = ',')

In [12]:

def safe_parse(val):
    """Convert a string representation of a list into an actual list."""
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return []
    return []

basket_cleaned = (
    basket
    .groupby('customer_id')['list_of_goods']
    .apply(lambda x: list(set(
        product
        for transaction in x                # iterate over each transaction string
        for product in safe_parse(transaction)  # unpack products from each string
    )))
    .reset_index()
)
basket_cleaned.columns = ['customer_id', 'list_of_goods']
basket_cleaned

,customer_id,list_of_goods
0,3,"[extra dark chocolate, black tea, cream, green..."
1,4,"[eggs, honey, energy drink, extra dark chocola..."
2,5,"[pickles, brownies, final fantasy XXII, brambl..."
3,7,"[deodorant, salt, cream, energy bar]"
4,9,"[deodorant, pokemon sword, bluetooth headphone..."
...,...,...
28122,39992,"[energy drink, extra dark chocolate, mineral w..."
28123,39994,"[energy drink, extra dark chocolate, mineral w..."
28124,39998,"[final fantasy XXII, spinach, megaman zero 3, ..."
28125,39999,"[gums, shampoo, black tea, final fantasy XXII,..."


In [13]:
print(basket[basket['customer_id']==3])

       invoice_id                                      list_of_goods  \
39973     7994762  ['burgers', 'tomatoes', 'megaman zero 3', 'muf...   
82563     5829775  ['cream', 'portal', 'green tea', 'pepper', 'so...   

       customer_id  
39973            3  
82563            3  


In [14]:
merged_df = pd.merge(costumer_info, basket_cleaned, on="customer_id", how="left")
merged_df

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude,list_of_goods
0,3,Bsc. Crystal Kitchens,female,1970-02-12 13:36:00,0.0,0.0,0.0,0.0,-0.104564,0.973784,...,-0.176781,0.132530,0.271429,0.461538,1.130011,0.714286,0.0,1.455541,-1.730469,"[extra dark chocolate, black tea, cream, green..."
1,4,Bsc. Glenda Bauman,female,1975-11-13 18:58:00,0.0,-1.0,-1.0,-0.5,0.056867,-0.160139,...,1.575198,0.441767,1.609524,0.048951,-0.258072,-0.285714,0.0,0.108044,-0.671733,"[eggs, honey, energy drink, extra dark chocola..."
2,5,Msc. Antonio Campbell,male,1971-09-10 10:07:00,-1.0,-1.0,-0.8,-0.5,-0.048972,-0.464308,...,-0.236148,-0.489960,-0.500000,-0.293706,-0.490802,-1.428571,0.0,1.021787,-0.116230,"[pickles, brownies, final fantasy XXII, brambl..."
3,7,John Kelling,male,1982-10-23 11:20:00,-1.0,-1.0,1.0,-1.0,-0.453084,-0.115287,...,-0.265172,5.755020,-0.680952,-0.216783,0.040803,0.857143,0.0,-0.275625,0.234757,"[deodorant, salt, cream, energy bar]"
4,8,Arthur Dematteo,male,1969-08-04 22:22:00,-1.0,-1.0,2.0,-1.0,-0.313775,2.959886,...,-0.513193,4.156627,0.542857,-0.818182,-0.152380,0.857143,0.0,-0.479940,-0.923065,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33033,39996,Joshua Howard,male,1973-05-16 14:04:00,-1.0,-1.0,1.0,-1.0,-1.030222,3.795957,...,-0.823219,5.522088,0.185714,0.272727,0.063115,-0.142857,0.0,-0.188653,-0.213398,NaN
33034,39997,Anthony Hines,male,1955-05-10 01:19:00,0.0,-1.0,0.0,-1.0,-0.376028,4.086545,...,-0.614776,7.558233,-0.476190,-0.181818,-0.276545,-0.142857,0.0,0.006907,-1.077141,NaN
33035,39998,Edna Hasselman,female,1945-05-15 23:09:00,0.0,0.0,-1.0,0.5,-0.409005,0.003159,...,1.362797,-0.538153,-0.328571,0.580420,-0.165783,-0.428571,0.0,0.874318,0.549367,"[final fantasy XXII, spinach, megaman zero 3, ..."
33036,39999,George Kramer,male,1951-05-25 21:02:00,0.0,0.0,0.0,1.5,-0.381456,0.519267,...,-0.364116,0.703614,-0.485714,1.832168,1.636190,0.285714,0.0,-0.386478,0.335687,"[gums, shampoo, black tea, final fantasy XXII,..."


In [15]:
import ast
from collections import Counter

def parse_goods(val):
    if isinstance(val, float):  # NaN
        return []
    
    # If it's already a proper list
    if isinstance(val, list):
        flat = []
        for item in val:
            if isinstance(item, list):
                flat.extend([str(p).strip() for p in item])
            else:
                flat.append(str(item).strip())
        return flat
    
    if isinstance(val, str):
        val = val.strip()
        if not val:
            return []
        try:
            parsed = ast.literal_eval(val)
            flat = []
            for item in parsed:
                if isinstance(item, list):
                    flat.extend([str(p).strip() for p in item])
                else:
                    flat.append(str(item).strip())
            return flat
        except Exception as e:
            print(f"Failed to parse: {repr(val)} → {e}")
            return []
    
    return []
# Flatten all products per client (with duplicates, for counting)
merged_df['all_products'] = merged_df['list_of_goods'].apply(parse_goods)

# Unique products per client
merged_df['unique_products'] = merged_df['all_products'].apply(
    lambda products: list(set(products))
)

# Count how many times each product appears
merged_df['product_counts'] = merged_df['all_products'].apply(
    lambda products: dict(Counter(products))
)

# Drop the helper column
merged_df.drop(columns=['all_products'], inplace=True)

#merged_df[['unique_products', 'product_counts']].head()
merged_df.head()

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude,list_of_goods,unique_products,product_counts
0,3,Bsc. Crystal Kitchens,female,1970-02-12 13:36:00,0.0,0.0,0.0,0.0,-0.104564,0.973784,...,0.271429,0.461538,1.130011,0.714286,0.0,1.455541,-1.730469,"[extra dark chocolate, black tea, cream, green...","[extra dark chocolate, black tea, cream, green...","{'extra dark chocolate': 1, 'black tea': 1, 'c..."
1,4,Bsc. Glenda Bauman,female,1975-11-13 18:58:00,0.0,-1.0,-1.0,-0.5,0.056867,-0.160139,...,1.609524,0.048951,-0.258072,-0.285714,0.0,0.108044,-0.671733,"[eggs, honey, energy drink, extra dark chocola...","[eggs, honey, energy drink, extra dark chocola...","{'eggs': 1, 'honey': 1, 'energy drink': 1, 'ex..."
2,5,Msc. Antonio Campbell,male,1971-09-10 10:07:00,-1.0,-1.0,-0.8,-0.5,-0.048972,-0.464308,...,-0.500000,-0.293706,-0.490802,-1.428571,0.0,1.021787,-0.116230,"[pickles, brownies, final fantasy XXII, brambl...","[pickles, tea, brownies, final fantasy XXII, b...","{'pickles': 1, 'brownies': 1, 'final fantasy X..."
3,7,John Kelling,male,1982-10-23 11:20:00,-1.0,-1.0,1.0,-1.0,-0.453084,-0.115287,...,-0.680952,-0.216783,0.040803,0.857143,0.0,-0.275625,0.234757,"[deodorant, salt, cream, energy bar]","[deodorant, salt, cream, energy bar]","{'deodorant': 1, 'salt': 1, 'cream': 1, 'energ..."
4,8,Arthur Dematteo,male,1969-08-04 22:22:00,-1.0,-1.0,2.0,-1.0,-0.313775,2.959886,...,0.542857,-0.818182,-0.152380,0.857143,0.0,-0.479940,-0.923065,NaN,[],{}


In [16]:
from hierachichal import KmeansClustering
data_for_clustering = merged_df.iloc[:, 4:-3].copy()

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_for_clustering)

In [17]:

km = KmeansClustering(
    min_k=2, max_k=8 + 3,
    data=data_scaled,
    random_seed=42,
)

labels, inertia, centroids = km.cluster(k=8, epochs=20)

In [31]:
# Assuming 'model' is your fitted KMeans or AgglomerativeClustering object
# and 'merged_df' is the table you showed me earlier.
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

merged_df['cluster_label'] =labels
merged_df

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,typical_hour,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,lifetime_spend_alcohol_drinks,lifetime_spend_meat,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude,list_of_goods,unique_products,product_counts,cluster_label,cluster_id
0,3,Bsc. Crystal Kitchens,female,1970-02-12 13:36:00,0.0,0.0,0.0,0.0,-0.104564,0.973784,0.075,-0.115294,-0.245614,-0.443318,-1.050975,-0.396277,-0.176781,0.132530,0.271429,0.461538,1.130011,0.714286,0.0,1.455541,-1.730469,"[extra dark chocolate, black tea, cream, green...","[extra dark chocolate, black tea, cream, green...","{'extra dark chocolate': 1, 'black tea': 1, 'c...",2,2
1,4,Bsc. Glenda Bauman,female,1975-11-13 18:58:00,0.0,-1.0,-1.0,-0.5,0.056867,-0.160139,0.100,1.812941,0.280702,-0.562115,-1.028486,-0.659574,1.575198,0.441767,1.609524,0.048951,-0.258072,-0.285714,0.0,0.108044,-0.671733,"[eggs, honey, energy drink, extra dark chocola...","[eggs, honey, energy drink, extra dark chocola...","{'eggs': 1, 'honey': 1, 'energy drink': 1, 'ex...",3,3
2,5,Msc. Antonio Campbell,male,1971-09-10 10:07:00,-1.0,-1.0,-0.8,-0.5,-0.048972,-0.464308,-0.125,0.098824,-0.802005,-0.528794,0.803598,-0.316489,-0.236148,-0.489960,-0.500000,-0.293706,-0.490802,-1.428571,0.0,1.021787,-0.116230,"[pickles, brownies, final fantasy XXII, brambl...","[pickles, tea, brownies, final fantasy XXII, b...","{'pickles': 1, 'brownies': 1, 'final fantasy X...",5,5
3,7,John Kelling,male,1982-10-23 11:20:00,-1.0,-1.0,1.0,-1.0,-0.453084,-0.115287,0.750,-0.455294,0.842105,0.941688,0.364318,0.760638,-0.265172,5.755020,-0.680952,-0.216783,0.040803,0.857143,0.0,-0.275625,0.234757,"[deodorant, salt, cream, energy bar]","[deodorant, salt, cream, energy bar]","{'deodorant': 1, 'salt': 1, 'cream': 1, 'energ...",4,4
4,8,Arthur Dematteo,male,1969-08-04 22:22:00,-1.0,-1.0,2.0,-1.0,-0.313775,2.959886,0.625,-0.107059,0.428571,0.340456,0.508246,0.670213,-0.513193,4.156627,0.542857,-0.818182,-0.152380,0.857143,0.0,-0.479940,-0.923065,NaN,[],{},4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33033,39996,Joshua Howard,male,1973-05-16 14:04:00,-1.0,-1.0,1.0,-1.0,-1.030222,3.795957,0.850,-0.330588,0.380952,0.740311,0.436282,0.855053,-0.823219,5.522088,0.185714,0.272727,0.063115,-0.142857,0.0,-0.188653,-0.213398,NaN,[],{},4,4
33034,39997,Anthony Hines,male,1955-05-10 01:19:00,0.0,-1.0,0.0,-1.0,-0.376028,4.086545,1.125,-0.211765,0.190476,1.353133,0.890555,0.121011,-0.614776,7.558233,-0.476190,-0.181818,-0.276545,-0.142857,0.0,0.006907,-1.077141,NaN,[],{},4,4
33035,39998,Edna Hasselman,female,1945-05-15 23:09:00,0.0,0.0,-1.0,0.5,-0.409005,0.003159,0.250,1.328235,0.182957,-0.530243,-0.632684,-0.579787,1.362797,-0.538153,-0.328571,0.580420,-0.165783,-0.428571,0.0,0.874318,0.549367,"[final fantasy XXII, spinach, megaman zero 3, ...","[final fantasy XXII, spinach, megaman zero 3, ...","{'final fantasy XXII': 1, 'spinach': 1, 'megam...",3,3
33036,39999,George Kramer,male,1951-05-25 21:02:00,0.0,0.0,0.0,1.5,-0.381456,0.519267,-0.125,0.147059,-0.295739,0.257878,-0.239280,-0.529255,-0.364116,0.703614,-0.485714,1.832168,1.636190,0.285714,0.0,-0.386478,0.335687,"[gums, shampoo, black tea, final fantasy XXII,...","[gums, shampoo, cat food, black tea, final fan...","{'gums': 1, 'shampoo': 1, 'black tea': 1, 'fin...",2,2


In [23]:
from collections import Counter

# Group by cluster and combine all dictionaries
for cluster in range(1, 8 + 1):
    cluster_data = merged_df[merged_df['cluster_label'] == cluster]
    
    # Sum up all the Counter dictionaries in this cluster
    total_basket = Counter()
    for d in cluster_data['product_counts']:
        total_basket.update(d)
    
    print(f"\n--- Top 50 Products in Segment {cluster} ---")
    print(total_basket.most_common(50))


--- Top 50 Products in Segment 1 ---
[('asparagus', 1899), ('tomatoes', 1453), ('carrots', 1447), ('spinach', 1443), ('avocado', 1432), ('salad', 1396), ('green beans', 1134), ('cauliflower', 1080), ('mineral water', 1064), ('green grapes', 1062), ('mashed potato', 1046), ('strawberries', 1040), ('zucchini', 1031), ('blueberries', 891), ('corn', 859), ('green tea', 855), ('antioxydant juice', 850), ('frozen vegetables', 825), ('eggplant', 800), ('mint', 797), ('shallot', 697), ('olive oil', 606), ('whole weat flour', 544), ('whole wheat rice', 539), ('cooking oil', 497), ('oil', 480), ('airpods', 419), ('gums', 386), ('salt', 379), ('fresh bread', 374), ('gluten free bar', 372), ('brownies', 361), ('almonds', 355), ('low fat yogurt', 352), ('butter', 352), ('herb & pepper', 351), ('whole wheat pasta', 349), ('melons', 348), ('energy bar', 345), ('turkey', 344), ('pet food', 344), ('ketchup', 343), ('french fries', 343), ('water spray', 342), ('honey', 341), ('muffins', 341), ('chili',

In [35]:
final_df=merged_df[['customer_id','unique_products','cluster_label']]
final_df

,customer_id,unique_products,cluster_label
0,3,"[extra dark chocolate, black tea, cream, green...",2
1,4,"[eggs, honey, energy drink, extra dark chocola...",3
2,5,"[pickles, tea, brownies, final fantasy XXII, b...",5
3,7,"[deodorant, salt, cream, energy bar]",4
4,8,[],4
...,...,...,...
33033,39996,[],4
33034,39997,[],4
33035,39998,"[final fantasy XXII, spinach, megaman zero 3, ...",3
33036,39999,"[gums, shampoo, cat food, black tea, final fan...",2


In [36]:
# 1. Add your cluster labels to the original merged_df
merged_df['cluster_label'] = labels

# 2. Select only the numeric columns and the cluster_id
# We exclude the list/dictionary columns for the mean calculation
numeric_cols = merged_df.iloc[:, 4:-3].columns.tolist() + ['cluster_label']

# 3. Group by cluster and calculate the mean
cluster_profiles = merged_df[numeric_cols].groupby('cluster_label').mean(numeric_only=True)

# Display the table
display(cluster_profiles)

,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,typical_hour,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,lifetime_spend_alcohol_drinks,lifetime_spend_meat,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
cluster_label,,,,,,,,,,,,,,,,,,,,,
1,0.809065,-0.269937,1.332415,-0.378485,-0.139945,-0.124115,-0.206605,-0.210786,-0.431469,0.127104,-0.167329,-0.006565,-0.287394,-0.273044,-0.103483,-0.362439,0.631242,0.285345,0.0,-0.011271,-0.062107
2,-0.382221,-0.491963,-0.304885,0.711133,-0.076430,-0.137147,0.383677,-0.043775,-0.422244,-0.033825,-0.047278,-0.114705,-0.258405,0.058054,-0.129355,-0.090434,0.968837,0.281373,0.0,0.004305,0.123252
3,0.040165,-0.143168,-0.425913,0.316343,-0.063175,-0.197426,-0.164900,1.192657,0.473450,-0.443888,-0.835333,-0.511463,1.709775,-0.284662,0.418015,0.007277,-0.245506,0.223835,0.0,0.479032,0.867449
4,-0.759842,-0.691086,0.078004,-0.786979,-0.353453,2.775201,0.699308,-0.345614,0.334257,0.885663,0.275518,0.467410,-0.569396,5.184851,-0.102892,-0.116422,0.050607,0.337346,0.0,0.003714,-0.462228
5,-0.413768,-0.167794,-0.693943,-0.612430,0.387595,-0.140687,-0.507139,0.149256,-0.280566,-0.336737,0.116647,0.252330,0.149397,-0.096603,0.401137,0.199302,-0.241319,-0.247038,0.0,-0.011532,-0.004741
6,2.337644,1.720858,-0.079764,0.098943,0.984229,1.133527,-0.242097,0.429846,1.012305,1.553907,1.174433,1.091082,0.305118,0.805679,0.171864,1.260995,-0.098828,-0.251210,0.0,-0.000482,-0.001761
7,0.033324,-0.104066,-0.224927,0.098352,-0.193529,0.169877,0.264144,1.781800,-0.008125,-0.094508,-0.978933,-0.558624,0.049813,0.051778,-0.316166,-0.280469,-0.402482,-0.013971,0.0,-0.035237,-0.028684
8,0.027407,-0.135093,0.075127,0.508552,1.551314,0.579948,0.185584,-0.127830,0.694277,0.377063,0.425623,0.601731,0.624990,0.444198,0.133254,0.934314,0.430208,-0.316090,0.0,0.003908,0.026916


In [38]:
# 1. Create a dictionary to store the top 50 item names for each cluster
cluster_top_50_map = {}

# Use your existing loop to populate the map
for cluster in range(1, 8 + 1):
    cluster_data = merged_df[merged_df['cluster_label'] == cluster]
    
    total_basket = Counter()
    for d in cluster_data['product_counts']:
        total_basket.update(d)
    
    # Store only the product names (not the counts) in a set for fast searching
    top_50_names = {item for item, count in total_basket.most_common(50)}
    cluster_top_50_map[cluster] = top_50_names

# 2. Define the scoring function
def get_affinity_score(row):
    # Convert unique_products to a set for comparison
    customer_items = set(row['unique_products'])
    if not customer_items:
        return 0.0
    
    # Get the Top 50 set for this specific client's cluster
    top_50_for_cluster = cluster_top_50_map.get(row['cluster_label'], set())
    
    # Count how many of the client's products are in the cluster's Top 50
    overlap = customer_items.intersection(top_50_for_cluster)
    
    # Return percentage: (items in top 50 / total unique items bought by client) * 100
    return (len(overlap) / len(customer_items)) * 100

# 3. Apply to the DataFrame
merged_df['cluster_affinity_score'] = merged_df.apply(get_affinity_score, axis=1)

# Check the results
nfinal=merged_df[['customer_id', 'unique_products','cluster_label','cluster_affinity_score']].head()
nfinal

,customer_id,unique_products,cluster_label,cluster_affinity_score
0,3,"[extra dark chocolate, black tea, cream, green...",2,33.333333
1,4,"[eggs, honey, energy drink, extra dark chocola...",3,40.000000
2,5,"[pickles, tea, brownies, final fantasy XXII, b...",5,37.500000
3,7,"[deodorant, salt, cream, energy bar]",4,75.000000
4,8,[],4,0.000000
